# 02 - Gate Intuition (X, Z, CX, CCX)

This notebook builds intuition for four foundational gates:
- X (bit flip)
- Z (phase flip)
- CX (controlled X)
- CCX / Toffoli (double-controlled X)

Prediction-first workflow: before each run, write your expected dominant output.

In [12]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator

In [13]:
def run_counts(circuit: QuantumCircuit, shots: int = 1024) -> dict[str, int]:
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    return backend.run(compiled, shots=shots).result().get_counts(compiled)

def probs(counts: dict[str, int]) -> dict[str, float]:
    total = sum(counts.values())
    return {k: v / total for k, v in sorted(counts.items())}

## X Gate: Bit Flip

Prediction: applying X to $|0\rangle$ should produce $|1\rangle$ with near certainty.

In [ ]:
# QuantumCircuit(num_qubits, num_classical_bits)
x_circuit = QuantumCircuit(1, 1)

# Apply X to qubit 0, which flips |0> to |1>.
x_circuit.x(0)

# Measure qubit 0 into classical bit 0.
# Indices must be in range for their registers.
x_circuit.measure(0, 0)

x_circuit.draw(output='text')

┌───┐┌─┐
  q: ┤ X ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0

In [11]:
x_counts = run_counts(x_circuit, shots=1024)
x_counts, probs(x_counts)

({'1': 1024}, {'1': 1.0})

## Z Gate: Phase Flip (Often Invisible in Direct Measurement)

On basis state $|0\rangle$, Z does not change measurement outcome.

Prediction: dominant result remains `0`.

In [ ]:
# One qubit and one classical bit for readout.
z_circuit = QuantumCircuit(1, 1)

# Apply Z to qubit 0 (phase flip, not a direct bit flip).
z_circuit.z(0)

# Measure qubit 0 into classical bit 0.
z_circuit.measure(0, 0)

z_circuit.draw(output='text')

┌───┐┌─┐
  q: ┤ Z ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0

In [ ]:
z_counts = run_counts(z_circuit, shots=1024)
z_counts, probs(z_counts)

## Making Z Visible: H-Z-H

$H Z H = X$ for a single qubit, so this sequence should behave like a bit flip on $|0\rangle$.

Prediction: dominant result becomes `1`.

In [ ]:
# One qubit and one classical bit.
hzh_circuit = QuantumCircuit(1, 1)

# First H puts the qubit into superposition.
hzh_circuit.h(0)

# Z changes relative phase in that superposition.
hzh_circuit.z(0)

# Final H converts phase difference into a bit-value difference.
hzh_circuit.h(0)

# Measure qubit 0 into classical bit 0.
hzh_circuit.measure(0, 0)

hzh_circuit.draw(output='text')

┌───┐┌───┐┌───┐┌─┐
  q: ┤ H ├┤ Z ├┤ H ├┤M├
     └───┘└───┘└───┘└╥┘
c: 1/════════════════╩═
                     0

In [ ]:
hzh_counts = run_counts(hzh_circuit, shots=1024)
hzh_counts, probs(hzh_counts)

## CX Gate: Controlled Bit Flip

Rule: flip target only if control is `1`.

We prepare qubit 1 as control in $|1\rangle$ and use qubit 0 as target.
Prediction: output should be dominated by `11`.

In [ ]:
# Two qubits and two classical bits.
cx_circuit = QuantumCircuit(2, 2)

# Prepare control qubit (index 1) in |1>.
cx_circuit.x(1)

# CX(control=1, target=0): flip target only when control is 1.
cx_circuit.cx(1, 0)

# Measure qubits [0, 1] into classical bits [0, 1] one-to-one.
cx_circuit.measure([0, 1], [0, 1])

cx_circuit.draw(output='text')

┌───┐┌─┐   
q_0: ─────┤ X ├┤M├───
     ┌───┐└─┬─┘└╥┘┌─┐
q_1: ┤ X ├──■───╫─┤M├
     └───┘      ║ └╥┘
c: 2/═══════════╩══╩═
                0  1

In [ ]:
cx_counts = run_counts(cx_circuit, shots=1024)
cx_counts, probs(cx_counts)

## CCX (Toffoli): Double-Controlled Flip

Rule: flip target only if both controls are `1`.

We prepare both controls as `1`, so target should flip to `1`.
Prediction: output should be dominated by `111`.

In [ ]:
# Three qubits and three classical bits.
ccx_circuit = QuantumCircuit(3, 3)

# Prepare both control qubits in |1>.
ccx_circuit.x(2)
ccx_circuit.x(1)

# CCX(control1=2, control2=1, target=0).
# Target flips only if both controls are 1.
ccx_circuit.ccx(2, 1, 0)

# Measure qubits [0, 1, 2] into matching classical bits.
ccx_circuit.measure([0, 1, 2], [0, 1, 2])

ccx_circuit.draw(output='text')

┌───┐┌─┐      
q_0: ─────┤ X ├┤M├──────
     ┌───┐└─┬─┘└╥┘┌─┐   
q_1: ┤ X ├──■───╫─┤M├───
     ├───┤  │   ║ └╥┘┌─┐
q_2: ┤ X ├──■───╫──╫─┤M├
     └───┘      ║  ║ └╥┘
c: 3/═══════════╩══╩══╩═
                0  1  2

In [ ]:
ccx_counts = run_counts(ccx_circuit, shots=1024)
ccx_counts, probs(ccx_counts)

## Reflection

1. In your own words, what is the difference between a bit flip (X) and a phase flip (Z)?
2. Why can Z look invisible when you directly measure basis states like $|0\rangle$ and $|1\rangle$?
3. For CX, which qubit is the control and what exact condition flips the target?
4. For CCX, what extra condition is required compared to CX?
5. In one sentence, explain why "same probabilities" does not always mean "same quantum state".

## Bonus Notes and Answer Guide

Use this as a reference after you attempt the reflection questions.

### 1) What do amplitudes mean in $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$?

- $\alpha$ and $\beta$ are amplitudes (state coordinates in the computational basis).
- They are not probabilities directly.
- Measurement probabilities are $|\alpha|^2$ and $|\beta|^2$.

### 2) Why must $|\alpha|^2 + |\beta|^2 = 1$?

- Total measurement probability must be 1.
- The qubit must collapse to one of the basis outcomes when measured.

### 3) Why can these two states look similar but behave differently later?

- $\frac{|0\rangle + |1\rangle}{\sqrt{2}}$
- $\frac{|0\rangle - |1\rangle}{\sqrt{2}}$

Both give 50/50 in direct Z-basis measurement, but they differ by relative phase (plus vs minus). Later gates can convert that phase difference into different output probabilities.

### 4) What does a sign change from Z actually do?

- Z keeps the $|0\rangle$ amplitude the same.
- Z flips the sign of the $|1\rangle$ amplitude.
- This phase information can affect interference in later gates.

### 5) What is interference in circuits?

- Constructive interference: amplitudes add, making an outcome more likely.
- Destructive interference: amplitudes cancel, making an outcome less likely.
- Quantum algorithms shape these adds and cancels to boost correct answers.

### Quick Check Examples

1. Why does H-Z-H behave like X on $|0\rangle$?

Because $H Z H = X$ for a single qubit, so phase change in the middle is converted into a bit flip by the surrounding H gates.

2. Why does CX only flip sometimes?

Because CX flips the target only when the control qubit is 1.

3. How is CCX stricter than CX?

CCX requires two controls to be 1 before flipping the target.